In [1]:
from pathlib import Path

#root_dir = Path('/work/moralization')
root_dir = Path.home() / 'scratch/projects/moralization'

In [2]:
import json

with (root_dir / 'data/reannotated_20251018.json').open('r') as f:
    data = json.load(f)
data = {d['text_id']: d for d in data}
len(data)

11555

contains_moralization

In [3]:
#for text_id, d in data.items():
def validate_binary_label(text_id, d):
    assert isinstance(d['contains_moralization'], bool), ('wrong dtype 1', text_id)
    assert isinstance(d['contains_moralization_original'], bool), ('wrong dtype 2', text_id)
    assert d['contains_moralization_reannotated'] in [True, False, None], ('wrong dtype 3', text_id)

kat1

In [4]:
kat1 = [
    'Keine Moralisierung',
    'Moralisierung explizit',
    'Moralisierung interpretativ',
    'Moralisierung Kontext',
    'Moralisierung Weltwissen',
    'Moralisierung fremde Stimme',
    'Doppelung',
    #'Nicht annotiert',
]
#for text_id, d in data.items():
def validate_kat1(text_id, d):
    # dtype
    assert isinstance(d['kat1_type'], str), ('wrong dtype 1', text_id)
    assert isinstance(d['kat1_type_original'], str), ('wrong dtype 1', text_id)
    assert isinstance(d['kat1_type_reannotated'], str), ('wrong dtype 1', text_id)

    # values
    assert d['kat1_type'] in kat1, ('wrong value 1', text_id)
    assert d['kat1_type_original'] in kat1 + ['Nicht annotiert'], ('wrong value 2', text_id)
    assert d['kat1_type_reannotated'] in kat1 + [''], ('wrong value 3', text_id)

    # contains_moralization
    if d['contains_moralization']:
        assert d['kat1_type'] in [
            'Moralisierung explizit',
            'Moralisierung interpretativ',
            'Moralisierung Kontext',
            'Moralisierung Weltwissen',
            'Moralisierung fremde Stimme',
            'Doppelung'
        ], ('wrong value 4', text_id)
    else:
        assert d['kat1_type'] in ['Keine Moralisierung', 'Nicht annotiert'], ('wrong value 5', text_id)

    # demand
    #if d['contains_moralization']:
    #    if 'implicit' in d['kat5_demand']:
    #        assert d['kat1_type'] in ['Moralisierung interpretativ', 'Moralisierung Kontext', 'Moralisierung Weltwissen', 'Moralisierung fremde Stimme'], text_id
    #    elif 'explicit' in d['kat5_demand']:
    #        assert d['kat1_type'] in ['Moralisierung explizit', 'Moralisierung Kontext', 'Moralisierung Weltwissen', 'Moralisierung fremde Stimme'], text_id

In [5]:
kat2 = {
    "Fürsorge": "Care",                         #'care.virtue',
    "Schaden": "Harm",                          #'care.vice',
    "Fairness": "Fairness",                     #'fairness.virtue',
    "Betrug": "Cheating",                       #'fairness.vice',
    "Loyalität": "Loyality",                    #'loyalty.virtue',
    "Verrat": "Betrayal",                       #'loyalty.vice',
    "Autorität": "Authority",                   #'authority.virtue',
    "Untergrabung von Autorität": "Subversion", #'authority.vice',
    "Reinheit": "Purity",                       #'purity.virtue',
    "Verfall": "Degradation",                   #'purity.vice',
    "Freiheit": "Liberty",                      #'liberty.virtue',
    "Unterdrückung": "Opression",               #'liberty.vice',
}
#for text_id, d in data.items():
def validate_kat2(text_id, d):
    # dtype
    for slices in ['', '_slices']:
        for suffix in ['', '_original', '_continuous', '_reannotated']:
            key = f'kat2_moral_value{slices}{suffix}'
            assert isinstance(d[key], dict), ('wrong dtype 1', text_id)
            if len(d[key]) > 0:
                for k, v in d[key].items():
                    assert isinstance(k, str), ('wrong dtype 2',text_id, key)
                    assert isinstance(v, list), ('wrong dtype 3',text_id, key)
                    if slices == '':
                        if len(v) > 0:
                            assert isinstance(v[0], str), ('wrong dtype 4',text_id, key)
                    else:
                        if len(v) > 0:
                            assert isinstance(v[0], list), ('wrong dtype 5',text_id, key)
                            assert len(v[0]) == 2, ('wrong dtype 6',text_id, key)
                            assert isinstance(v[0][0], int), ('wrong dtype 7',text_id, key) # start
                            assert isinstance(v[0][1], int), ('wrong dtype 8',text_id, key) # end

    # values
    if d['contains_moralization']:
        assert len(d['kat2_moral_value']) > 0, ('wrong value 1', text_id)
        for label, surfaces in d['kat2_moral_value'].items():
            assert label in kat2, ('wrong value 2', text_id, label)
            slices = d['kat2_moral_value_slices'][label]
            assert len(set([tuple(s) for s in slices])) == len(slices), ('wrong value 3', text_id, label) # slices must be unique
            assert len(surfaces) == len(slices), ('wrong value 4', text_id, label)

            for phrase, span in zip(surfaces, slices):
                assert len(span)
                start, end = span
                assert d['text'][start:end] == phrase, ('wrong value 5', text_id, label, phrase, d['text'][start:end], start, end)
    else:
        assert len(d['kat2_moral_value']) == 0, ('wrong value 6', text_id)
        assert len(d['kat2_moral_value_slices']) == 0, ('wrong value 7', text_id)

kat3

In [6]:
kat3 = [
    'Individuum',
    'Menschen',
    'Institution',
    'Soziale-Gruppe',
    'OTHER',
]
roles = [
    'Adressat:in',
    'Benefizient:in',
    'Forderer:in',
    'Malefizient:in',
    'Bezug-unklar',
]
#for text_id, d in data.items():
def validate_kat3(text_id, d):
    # dtype
    for slices in ['', '_slices']:
        for suffix in ['', '_original', '_continuous', '_reannotated']:
            key = f'kat3_protagonist{slices}{suffix}'
            assert isinstance(d[key], dict), ('wrong dtype 1', text_id)
            if len(d[key]) > 0:
                for k, v in d[key].items():
                    assert isinstance(k, str), ('wrong dtype 2',text_id, key)
                    assert isinstance(v, list), ('wrong dtype 3',text_id, key)
                    if slices == '':
                        if len(v) > 0:
                            assert isinstance(v[0], str), ('wrong dtype 4', text_id, key)
                    else:
                        if len(v) > 0:
                            assert isinstance(v[0], list), ('wrong dtype 5', text_id, key)
                            assert len(v[0]) == 2, ('wrong dtype 6', text_id, key)
                            assert isinstance(v[0][0], int), ('wrong dtype 7', text_id, key) # start
                            assert isinstance(v[0][1], int), ('wrong dtype 8', text_id, key) # end

    # values
    if d['contains_moralization']:
        #assert len(d['kat3_protagonist']) > 0, text_id # can be empty!
        for label, surfaces in d['kat3_protagonist'].items():
            role, protagonist = label.split('_')
            assert protagonist in kat3, ('wrong value 1', text_id, protagonist, label)
            assert role in roles, ('wrong value 2', text_id, role, label)
            slices = d['kat3_protagonist_slices'][label]
            assert len(set([tuple(s) for s in slices])) == len(slices), ('wrong value 3', text_id, label) # slices must be unique
            assert len(surfaces) == len(slices), ('wrong value 4', text_id, label)

            for phrase, span in zip(surfaces, slices):
                assert len(span)
                start, end = span
                assert d['text'][start:end] == phrase, ('wrong value 5', text_id, label, phrase, d['text'][start:end], start, end)
    else:
        assert len(d['kat3_protagonist']) == 0, ('wrong value 6', text_id)
        assert len(d['kat3_protagonist_slices']) == 0, ('wrong value 7', text_id)


kat4

In [7]:
kat4 = [
    'Appell',
    'Appell+Darstellung',
    'Appell+Expression',
    'Appell+Beziehung',
]

#for text_id, d in data.items():
def validate_kat4(text_id, d):
    # dtype
    for slices in ['', '_slices']:
        for suffix in ['', '_original', '_reannotated']:
            key = f'kat4_communicative_function{slices}{suffix}'
            assert isinstance(d[key], dict), ('wrong dtype 1', text_id)
            if len(d[key]) > 0:
                for k, v in d[key].items():
                    assert isinstance(k, str), ('wrong dtype 2', text_id, key)
                    assert isinstance(v, list), ('wrong dtype 3', text_id, key)
                    if slices == '':
                        if len(v) > 0:
                            assert isinstance(v[0], str), ('wrong dtype 4', text_id, key)
                    else:
                        if len(v) > 0:
                            assert isinstance(v[0], list), ('wrong dtype 5', text_id, key)
                            assert len(v[0]) == 2, (text_id, key)
                            assert isinstance(v[0][0], int), ('wrong dtype 6', text_id, key) # start
                            assert isinstance(v[0][1], int), ('wrong dtype 7', text_id, key) # end

    # values
    if d['contains_moralization']:
        #assert len(d['kat4_communicative_function']) > 0, text_id # can be empty!
        for label, surfaces in d['kat4_communicative_function'].items():
            assert label in kat4, ('wrong value 1', text_id, label)
            slices = d['kat4_communicative_function_slices'][label]
            assert len(set([tuple(s) for s in slices])) == len(slices), ('wrong value 2', text_id, label) # slices must be unique
            assert len(surfaces) == len(slices), ('wrong value 3', text_id, label)

            for phrase, span in zip(surfaces, slices):
                assert len(span)
                start, end = span
                assert d['text'][start:end] == phrase, ('wrong value 4', text_id, label, phrase, d['text'][start:end], start, end)
    else:
        assert len(d['kat4_communicative_function']) == 0, ('wrong value 5', text_id)
        assert len(d['kat4_communicative_function_slices']) == 0, ('wrong value 6', text_id)


kat5

In [8]:
#for text_id, d in data.items():
def validate_kat5(text_id, d):
    # dtype
    for slices in ['', '_slices']:
        for suffix in ['', '_original', '_reannotated']:
            key = f'kat5_demand{slices}{suffix}'
            assert isinstance(d[key], dict), ('wrong dtype 1', text_id)
            if len(d[key]) > 0:
                for k, v in d[key].items():
                    if isinstance(v, str):
                        d[key][k] = [v]
                    assert isinstance(k, str), ('wrong dtype 2', text_id, key)
                    assert isinstance(v, list), ('wrong dtype 3', text_id, key)
                    if slices == '':
                        if len(v) > 0:
                            assert isinstance(v[0], str), ('wrong dtype 4', text_id, key)
                    else:
                        if len(v) > 0:
                            assert isinstance(v[0], list), ('wrong dtype 5', text_id, key)
                            assert len(v[0]) == 2, (text_id, key)
                            assert isinstance(v[0][0], int), ('wrong dtype 6', text_id, key) # start
                            assert isinstance(v[0][1], int), ('wrong dtype 7', text_id, key) # end

    # values
    if d['contains_moralization']:
        assert len(d['kat5_demand']) > 0, ('wrong value 1', text_id)
        #assert not (len(d['kat5_demand']) > 1), ('wrong value 1', text_id, label) # at most 1
        for label, surfaces in d['kat5_demand'].items():
            assert label in ['implicit', 'explicit'], ('wrong value 2', text_id, label)
            if label == 'explicit':
                slices = d['kat5_demand_slices'][label]
                assert len(set([tuple(s) for s in slices])) == len(slices), ('wrong value 3', text_id, label) # slices must be unique
                assert len(surfaces) == len(slices), ('wrong value 4', text_id, label)

                for phrase, span in zip(surfaces, slices):
                    assert len(span)
                    start, end = span
                    assert d['text'][start:end] == phrase, ('wrong value 5', text_id, label, phrase, d['text'][start:end], start, end)
    else:
        assert len(d['kat5_demand']) == 0, ('wrong value 6', text_id)
        assert len(d['kat5_demand']) == 0, ('wrong value 7', text_id)

In [9]:
for text_id, d in data.items():
    #d['is_valid_kat1'] = True
    #d['is_valid_kat2+kat5'] = True
    #d['is_valid_kat3'] = True
    #d['is_valid_kat4'] = True

    try:
        validate_binary_label(text_id, d)
        validate_kat1(text_id, d)
    except Exception as e:
        print('invalid kat1', d['split'], e)
        d['is_valid_kat1'] = False

    try:
        validate_kat2(text_id, d)
        validate_kat5(text_id, d)
    except Exception as e:
        print('invalid kat2+5', d['split'], e)
        d['is_valid_kat2+kat5'] = False

    try:
        validate_kat3(text_id, d)
    except Exception as e:
        print('invalid kat3', d['split'], e)
        d['is_valid_kat3'] = False

    try:
        validate_kat4(text_id, d)
    except Exception as e:
        print('invalid kat4', d['split'], e)
        d['is_valid_kat4'] = False

    d['is_valid'] = d['is_valid_kat1'] and d['is_valid_kat2+kat5'] and d['is_valid_kat3'] and d['is_valid_kat4']

invalid kat2+5 test ('wrong value 1', 'them-Kommentare-neg-0886')
invalid kat2+5 test ('wrong value 1', 'mor-Leserbriefe-neg-0351')
invalid kat2+5 test ('wrong value 1', 'them-Leserbriefe-neg-0329')
invalid kat2+5 train 'explicit'
invalid kat4 train ('wrong value 2', 'mor-Plenar-neg-0252', 'Appell+Darstellung')
invalid kat4 dev 'Appell'
invalid kat4 dev ('wrong value 2', 'mor-Plenar-neg-0264', 'Appell')
invalid kat4 train ('wrong value 2', 'mor-Plenar-neg-0302', 'Appell')
invalid kat2+5 dev ('wrong value 5', 'mor-Plenar-neg-0308', 'explicit', 'lle erforderlichen Maßnahmen zur Rehabilitierung der Betroffenen einzuleiten, dafür einzutreten, dass Verfassungsschutzakten, die auf dem Radikalenerlass beruhen, den Betroffenen und der Wissenschaft zugänglich gemacht werden und dass gesetzliche Regelungen zur materiellen Entschädigung der Betroffenen geschaffen werden; die mit der Bewilligung von Mitteln aus den Programmen gegen Rechtsextremismus verbundene Extremismusklausel ersatzlos zu strei

In [10]:
for text_id, d in data.items():
    if d['split'] in ['test', 'test-150'] and (not d['is_valid']):
        print(text_id)

them-Kommentare-neg-0886
mor-Leserbriefe-neg-0351
them-Leserbriefe-neg-0329


prepare submission

In [11]:
keys = [
    'text_id', 'split', 'genre', 'text', 'source', 'contains_moralization',
    'kat1_type', 'kat2_moral_value', 'kat2_moral_value_slices',
    'kat3_protagonist', 'kat3_protagonist_slices',
    'kat4_communicative_function', 'kat4_communicative_function_slices',
    'kat5_demand', 'kat5_demand_slices',
    'annotator', 'annotator_comment', 'version'
]

In [12]:
lrec_data = []
for text_id, d in data.items():
    if d['split'] in ['test', 'test-150'] and d['is_valid']:
        lrec_data.append({k: d[k] for k in keys})

with (root_dir / 'data/testset_LREC2026.json').open('w') as f:
    json.dump(lrec_data, f, indent=4)

In [13]:
len(lrec_data)

1734

tables

In [14]:
splits = ["train", "dev", "test", "test-150"]
genre = ['Leserbriefe', 'Sachbücher', 'Kommentare', 'Wikipedia_Diskussionen', 'Interviews', 'Plenarprotokolle', 'Gerichtsurteile']
binary_label = ['moralisations','thematisations','ratio_moralisations/(moralisations+thematisations)']
kat1 = [
    'Keine Moralisierung',
    'Moralisierung explizit',
    'Moralisierung interpretativ',
    'Moralisierung Kontext',
    'Moralisierung Weltwissen',
    'Moralisierung fremde Stimme',
    'Doppelung',
    #'Nicht annotiert',
]
kat2 = {
    "Fürsorge":  "Care",
    "Schaden": "Harm",
    "Fairness": "Fairness",
    "Betrug": "Cheating",
    "Loyalität": "Loyality",
    "Verrat": "Betrayal",
    "Autorität": "Authority",
    "Untergrabung von Autorität": "Subversion",
    "Reinheit": "Purity",
    "Verfall": "Degradation",
    "Freiheit": "Liberty",
    "Unterdrückung": "Opression",
}
kat3_role = ["Forderer:in", "Adressat:in", "Benefizient:in", "Malefizient:in", "Bezug-unklar"]
kat3_group = ["Individuum", "Menschen", "Institution", "Soziale-Gruppe", "OTHER"]
kat4 = [
    'Appell',
    'Appell+Darstellung',
    'Appell+Expression',
    'Appell+Beziehung',
]
kat5 = ['explicit', 'implicit']

In [15]:
from collections import Counter

def print_table(data, json_field, label_map):
    c = Counter()
    for text_id, d in data.items():
        if d['is_valid']:
            target = d[json_field]
            if json_field in ['contains_moralization', 'genre', 'kat1_type']:
                c[(target, d['split'])] += 1
            elif json_field in ['kat2_moral_value']:
                for k, v in target.items():
                    c[(k, d['split'])] += len(v)
            elif json_field in ['kat3_protagonist']:
                for k, v in target.items():
                    role, group = k.split('_')
                    if role in label_map:
                        c[(role, d['split'])] += len(v)
                    elif group in label_map:
                        c[(group, d['split'])] += len(v)
            elif json_field in ['kat4_communicative_function', 'kat5_demand']:
                for k, v in target.items():
                    c[(k, d['split'])] += 1
    total_x = {split: sum([v for (l, s), v in c.items() if s == split]) for split in splits}
    total_y = {label: sum([v for (l, s), v in c.items() if l == label]) for label in label_map}
    
    print('|    |'+'|'.join(label_map.values())+'|total|')
    print('|:--:|'+'|'.join(['--:' for _ in label_map])+'|--:|')
    for s in splits + ['all']:
        print(f'|{s}|', end='')
        for key in label_map:
            value = total_y[key] if s == 'all' else c.get((key, s), 0)
            print(f'{value:d}|', end='')
        print(f'{total_x.get(s, sum(total_x.values())):d}|', end='')
        print()

In [16]:
print_table(data, 'contains_moralization', {True: 'Moralization', False: 'Thematization'})

|    |Moralization|Thematization|total|
|:--:|--:|--:|--:|
|train|1182|6634|7816|
|dev|294|1659|1953|
|test|489|1095|1584|
|test-150|49|101|150|
|all|2014|9489|11503|


In [17]:
print_table(data, 'genre', dict(zip(genre, genre)))

|    |Leserbriefe|Sachbücher|Kommentare|Wikipedia_Diskussionen|Interviews|Plenarprotokolle|Gerichtsurteile|total|
|:--:|--:|--:|--:|--:|--:|--:|--:|--:|
|train|1057|1161|1229|1126|1261|1077|905|7816|
|dev|263|291|307|283|315|266|228|1953|
|test|203|241|249|224|259|214|194|1584|
|test-150|30|16|22|26|20|26|10|150|
|all|1553|1709|1807|1659|1855|1583|1337|11503|


In [18]:
print_table(data, 'kat1_type', dict(zip(kat1, kat1)))

|    |Keine Moralisierung|Moralisierung explizit|Moralisierung interpretativ|Moralisierung Kontext|Moralisierung Weltwissen|Moralisierung fremde Stimme|Doppelung|total|
|:--:|--:|--:|--:|--:|--:|--:|--:|--:|
|train|6634|741|125|197|113|4|2|7816|
|dev|1659|177|39|47|31|0|0|1953|
|test|1095|209|105|92|74|9|0|1584|
|test-150|101|28|3|6|12|0|0|150|
|all|9489|1155|272|342|230|13|2|11503|


In [19]:
print_table(data, 'kat2_moral_value', kat2)

|    |Care|Harm|Fairness|Cheating|Loyality|Betrayal|Authority|Subversion|Purity|Degradation|Liberty|Opression|total|
|:--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|
|train|378|575|316|240|72|66|58|28|90|56|185|107|2171|
|dev|110|144|64|62|22|7|16|15|27|13|39|24|543|
|test|181|316|165|142|39|20|19|19|50|53|68|52|1124|
|test-150|24|46|23|11|5|13|3|1|6|9|5|3|149|
|all|693|1081|568|455|138|106|96|63|173|131|297|186|3987|


In [20]:
print_table(data, 'kat3_protagonist', dict(zip(kat3_role, kat3_role)))

|    |Forderer:in|Adressat:in|Benefizient:in|Malefizient:in|Bezug-unklar|total|
|:--:|--:|--:|--:|--:|--:|--:|
|train|495|717|718|19|234|2183|
|dev|129|176|178|6|58|547|
|test|189|366|380|127|129|1191|
|test-150|24|35|37|8|2|106|
|all|837|1294|1313|160|423|4027|


In [21]:
print_table(data, 'kat3_protagonist', dict(zip(kat3_group, kat3_group)))

|    |Individuum|Menschen|Institution|Soziale-Gruppe|OTHER|total|
|:--:|--:|--:|--:|--:|--:|--:|
|train|413|301|731|687|51|2183|
|dev|103|63|170|192|19|547|
|test|279|209|328|324|51|1191|
|test-150|19|14|49|19|5|106|
|all|814|587|1278|1222|126|4027|


In [22]:
print_table(data, 'kat4_communicative_function', dict(zip(kat4, kat4)))

|    |Appell|Appell+Darstellung|Appell+Expression|Appell+Beziehung|total|
|:--:|--:|--:|--:|--:|--:|
|train|302|691|274|35|1302|
|dev|77|178|63|11|329|
|test|126|268|93|24|511|
|test-150|11|30|9|3|53|
|all|516|1167|439|73|2195|


In [23]:
print_table(data, 'kat5_demand', dict(zip(kat5, kat5)))

|    |explicit|implicit|total|
|:--:|--:|--:|--:|
|train|648|534|1182|
|dev|158|136|294|
|test|235|254|489|
|test-150|28|21|49|
|all|1069|945|2014|
